<a href="https://colab.research.google.com/github/ms914/3DKWM_server/blob/main/KWM_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Kugelwolkenmodell-Mathematik. Erstellt JSON Objekt.

In [ ]:
import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List

app = FastAPI(title="Kugelwolkenmodell Backend-API")

# CORS erlauben, damit auch lokale Webseiten (Three.js) auf den Server zugreifen dürfen
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================================
# MATHEMATISCHER KERN (Aus den vorherigen Schritten)
# ==========================================
class Kugelwolke:
    def __init__(self, wolken_id, lokaler_vektor):
        self.id = wolken_id
        self.lokaler_vektor = np.array(lokaler_vektor, dtype=float)
        self.welt_vektor = np.zeros(3)
        self.elektronen = 0

class Atom:
    def __init__(self, uid, symbol, valenzelektronen, position=[0.0, 0.0, 0.0]):
        self.uid = uid
        self.symbol = symbol
        self.valenzelektronen = valenzelektronen
        self.position = np.array(position, dtype=float)
        self.rotation = np.eye(3)
        self.radius = 1.0
        self.wolken = []

        # Tetraeder-Geometrie
        basis = [np.array([1,1,1]), np.array([-1,-1,1]), np.array([-1,1,-1]), np.array([1,-1,-1])]
        for i, v in enumerate(basis):
            nv = (v / np.linalg.norm(v)) * self.radius
            self.wolken.append(Kugelwolke(i, nv))

        # Hund & Pauli Besetzung
        uebrig = self.valenzelektronen
        for w in self.wolken:
            if uebrig > 0: w.elektronen = 1; uebrig -= 1
        for w in self.wolken:
            if uebrig > 0: w.elektronen = 2; uebrig -= 1

        self.update_welt_koordinaten()

    def update_welt_koordinaten(self):
        for w in self.wolken:
            w.welt_vektor = (self.rotation @ w.lokaler_vektor) + self.position

    def to_dict(self):
        """Konvertiert das Atom in ein UI-freundliches JSON-Format."""
        return {
            "uid": self.uid,
            "symbol": self.symbol,
            "position": self.position.tolist(),
            "rotation": self.rotation.flatten().tolist(), # Als flaches Array für WebGL/Three.js
            "wolken": [
                {
                    "id": w.id,
                    "richtung_lokal": (w.lokaler_vektor / self.radius).tolist(),
                    "welt_position": w.welt_vektor.tolist(),
                    "elektronen": w.elektronen
                } for w in self.wolken
            ]
        }

# Procrustes & Bindungslogik
def berechne_procrustes_rotation(A, B):
    M = A @ B.T
    U, S, Vt = np.linalg.svd(M)
    R = U @ Vt
    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt
    return R

# Globaler Zustand der Simulation (Unsere "Single Source of Truth")
WELT_ZUSTAND = {
    "O_links": Atom("O_links", "O", 6, position=[0, 0, 0]),
    "O_rechts": Atom("O_rechts", "O", 6, position=[3, 2, 1]) # Verdreht/Verschoben im Raum
}

# ==========================================
# API-ENDPOINTS (DIE BRÜCKE ZUR UI)
# ==========================================

@app.get("/status")
def get_status():
    """Die UI ruft diesen Endpoint auf, um zu wissen, was sie zeichnen soll."""
    return {uid: atom.to_dict() for uid, atom in WELT_ZUSTAND.items()}


class BindungsAnfrage(BaseModel):
    atom_A_id: str
    wolken_A_ids: List[int]
    atom_B_id: str
    wolken_B_ids: List[int]

@app.post("/binde")
def binde_atome(anfrage: BindungsAnfrage):
    """Die UI sendet eine POST-Anfrage hierhin, wenn der Nutzer eine Bindung triggert."""
    if anfrage.atom_A_id not in WELT_ZUSTAND or anfrage.atom_B_id not in WELT_ZUSTAND:
        raise HTTPException(status_code=404, detail="Atom nicht gefunden")

    atom_A = WELT_ZUSTAND[anfrage.atom_A_id]
    atom_B = WELT_ZUSTAND[anfrage.atom_B_id]
    k = len(anfrage.wolken_A_ids)

    # Procrustes Matrizen aufbauen
    matrix_A = np.zeros((3, k))
    matrix_B = np.zeros((3, k))

    for i in range(k):
        w_A = atom_A.wolken[anfrage.wolken_A_ids[i]]
        w_B = atom_B.wolken[anfrage.wolken_B_ids[i]]

        if w_A.elektronen != 1 or w_B.elektronen != 1:
            raise HTTPException(status_code=400, detail="Nur einfach besetzte Wolken erlaubt!")

        matrix_A[:, i] = (w_A.welt_vektor - atom_A.position) / atom_A.radius
        matrix_B[:, i] = -(w_B.lokaler_vektor / atom_B.radius)

    # Transformation berechnen
    atom_B.rotation = berechne_procrustes_rotation(matrix_A, matrix_B)

    kern_abstand = 2.0 if k == 1 else (1.63 if k == 2 else 0.67)
    bindungs_achse = matrix_A[:, 0] if k == 1 else (matrix_A[:, 0] + matrix_A[:, 1]) / 2.0
    bindungs_achse /= np.linalg.norm(bindungs_achse)

    atom_B.position = atom_A.position + (bindungs_achse * kern_abstand)

    # Zustände auf "gebunden" (3) setzen
    for i in range(k):
        atom_A.wolken[anfrage.wolken_A_ids[i]].elektronen = 3
        atom_B.wolken[anfrage.wolken_B_ids[i]].elektronen = 3

    atom_A.update_welt_koordinaten()
    atom_B.update_welt_koordinaten()

    return {"status": "success", "message": f"{k}-fach Bindung erfolgreich hergestellt."}


@app.post("/reset")
def reset_simulation():
    """Setzt die Atome auf ihre ursprünglichen, ungebundenen Werte zurück."""
    global WELT_ZUSTAND

    # Atome komplett neu instanziieren (das setzt Position, Rotation und Elektronen zurück)
    WELT_ZUSTAND = {
        "O_links": Atom("O_links", "O", 6, position=[0, 0, 0]),
        "O_rechts": Atom("O_rechts", "O", 6, position=[3, 2, 1]) # Wieder verdreht/versetzt
    }

    return {"status": "success", "message": "Simulation zurückgesetzt."}

Server starten

In [ ]:
import multiprocessing
import uvicorn
import nest_asyncio

# Erlaubt verschachtelte Event-Loops in Jupyter/Colab Notebooks
nest_asyncio.apply()

def start_api():
    # Wir übergeben direkt das 'app'-Objekt aus Zelle 1
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Server in einen separaten Hintergrundprozess auslagern
server_prozess = multiprocessing.Process(target=start_api)
server_prozess.start()
print("Backend-Server wurde erfolgreich im Hintergrund auf Port 8000 gestartet!")

Backend-Server wurde erfolgreich im Hintergrund auf Port 8000 gestartet!


Test server

In [ ]:
#server_prozess.terminate()
#print("Server gestoppt.")

In [ ]:
# 1. Installiere pyngrok
!pip install pyngrok

from pyngrok import ngrok

# ==========================================
# HIER FÜGST DU DEINEN AUTHTOKEN EIN
# ==========================================
# Ersetze 'DEIN_NGROK_AUTHTOKEN_HIER' mit dem langen Code von der ngrok-Webseite
NGROK_TOKEN = "1Y1jBALsPqgA8RQOJd4HDib3Ofh_6HgoK1dPV8iBcKbVvxbQb"

# Dem ngrok-System den Token mitteilen
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Jetzt erst den Tunnel zu Port 8000 öffnen
public_url = ngrok.connect(8000, "http")
print("Deine öffentliche API-Zustandsadresse für deine lokale UI lautet:")
print(f"{public_url.public_url}/status")

Deine öffentliche API-Zustandsadresse für deine lokale UI lautet:
https://3f47-136-112-150-251.ngrok-free.app/status


In [ ]:
from fastapi.responses import HTMLResponse

@app.get("/", response_class=HTMLResponse)
def read_root():
    # 1. Lies die lokale index.html ein
    with open("index.html", "r", encoding="utf-8") as f:
        html_content = f.read()

    # 2. Wir suchen NUR nach dem Variablen-Namen und überschreiben die Zuweisung komplett.
    # Egal ob in deiner index.html 'http...' oder 'https...' steht:
    # Wir zwingen das JavaScript dazu, die aktuelle Server-Adresse dynamisch zu ermitteln.

    # Wir ersetzen die alte Zeile komplett mit einer sauberen JavaScript-Anweisung:
    html_content = html_content.replace(
        "const API_URL = 'http://127.0.0.1:8000';",
        "const API_URL = window.location.origin;"
    )

    # Falls du in der index.html schon die https-ngrok-Adresse fest eingetragen hattest,
    # fangen wir das hier sicherheitshalber auch noch ab:
    if "const API_URL = 'https://" in html_content:
        # Wir suchen das Ende der Zeile und ersetzen sie sauber
        start_idx = html_content.find("const API_URL = 'https://")
        end_idx = html_content.find("';", start_idx) + 2
        alte_zeile = html_content[start_idx:end_idx]
        html_content = html_content.replace(alte_zeile, "const API_URL = window.location.origin;")

    return html_content

index.html mit three.js

In [ ]:
# html and javascript
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <title>Kugelwolkenmodell 3D Viewport</title>
    <style>
        body { margin: 0; overflow: hidden; background-color: #1a1a1a; color: white; font-family: sans-serif; }
        #ui-layer { position: absolute; top: 10px; left: 10px; z-index: 10; background: rgba(0,0,0,0.8); padding: 15px; border-radius: 8px; }
        button { background: #2980b9; color: white; border: none; padding: 10px 15px; border-radius: 4px; cursor: pointer; font-weight: bold;}
        button:hover { background: #3498db; }
    </style>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
</head>
<body>



    <div id="ui-layer">
        <h2>KWM 3D-Simulation</h2>
        <p>Server-Status: <span id="server-status">Verbinde...</span></p>
        <button id="btn-binden">Doppelbindung ausführen (SVD)</button>
        <button id="btn-reset" style="background: #e74c3c; margin-top: 5px;">Reset Simulation</button>
    </div>

    <script>
        // ========================================================
        // CONFIG: HIER DEINE API-URL EINTRAGEN
        // ========================================================
        const API_URL = 'https://3f47-136-112-150-251.ngrok-free.app'; // Oder deine https://...ngrok-free.app

        // ========================================================
        // 1. THREE.JS INITIALISIERUNG (DIE BÜHNE)
        // ========================================================
        const scene = new THREE.Scene();
        const camera = new THREE.PerspectiveCamera(75, window.innerWidth / window.innerHeight, 0.1, 1000);
        const renderer = new THREE.WebGLRenderer({ antialias: true });
        renderer.setSize(window.innerWidth, window.innerHeight);
        document.body.appendChild(renderer.domElement);

        // Kamera-Steuerung mit der Maus
        const controls = new THREE.OrbitControls(camera, renderer.domElement);
        camera.position.set(0, 3, 7);
        controls.update();

        // Lichtquellen hinzufügen
        const ambientLight = new THREE.AmbientLight(0xffffff, 0.4);
        scene.add(ambientLight);
        const dirLight = new THREE.DirectionalLight(0xffffff, 0.8);
        dirLight.position.set(5, 10, 7);
        scene.add(dirLight);

        // Speicher für die 3D-Objekte der Atome, um sie später zu bewegen
        const atomMeshes = {};

        // ========================================================
        // 2. RENDERING LOGIK (AUS DATEN WERDEN KÖRPER)
        // ========================================================
        function aktualisiere3DSzene(daten) {
            for (const uid in daten) {
                const atomDaten = daten[uid];

                // Wenn das Atom noch nicht in der 3D-Szene existiert, erstelle es
                if (!atomMeshes[uid]) {
                    const atomGruppe = new THREE.Group();

                    // Atomkern (Rote Kugel für Sauerstoff)
                    const kernGeo = new THREE.SphereGeometry(0.3, 32, 32);
                    const kernMat = new THREE.MeshStandardMaterial({ color: 0xe74c3c, roughness: 0.3 });
                    const kernMesh = new THREE.Mesh(kernGeo, kernMat);
                    atomGruppe.add(kernMesh);

                    // Die 4 Kugelwolken als Sub-Elemente hinzufügen
                    atomGruppe.wolkenVisuals = [];
                    atomDaten.wolken.forEach(w => {
                        const wolkeGeo = new THREE.SphereGeometry(0.5, 32, 32); // Radius 0.5

                        // Farbe je nach Zustand (Elektronen) definieren
                        let farbe = 0x3498db; // Blau für freie reaktive Wolken
                        let opazitaet = 0.3;
                        if (w.elektronen === 2) { farbe = 0x95a5a6; opazitaet = 0.6; } // Grau für Paare

                        const wolkeMat = new THREE.MeshStandardMaterial({
                            color: farbe,
                            transparent: true,
                            opacity: opazitaet,
                            wireframe: false
                        });
                        const wolkeMesh = new THREE.Mesh(wolkeGeo, wolkeMat);

                        // Wolke lokal im Tetraeder-Abstand platzieren
                        wolkeMesh.position.fromArray(w.richtung_lokal);
                        atomGruppe.add(wolkeMesh);

                        // Referenz merken, um später die Farbe zu ändern
                        atomGruppe.wolkenVisuals[w.id] = wolkeMesh;
                    });

                    scene.add(atomGruppe);
                    atomMeshes[uid] = atomGruppe;
                }

                // AKTUALISIERUNG VON POSITION UND ROTATION AUS PYTHON
                const atomGruppe = atomMeshes[uid];
                atomGruppe.position.fromArray(atomDaten.position);

                // Rotationsmatrix (3x3) in Three.js Matrix4 konvertieren
                const m4 = new THREE.Matrix4();
                const r = atomDaten.rotation;
                m4.set(
                    r[0], r[1], r[2], 0,
                    r[3], r[4], r[5], 0,
                    r[6], r[7], r[8], 0,
                    0,    0,    0,    1
                );
                atomGruppe.rotation.setFromRotationMatrix(m4);

                // Update der Wolkenfarben, falls sie jetzt gebunden (Wert 3) sind
                atomDaten.wolken.forEach(w => {
                    if (w.elektronen === 3) {
                        atomGruppe.wolkenVisuals[w.id].material.color.setHex(0x2ecc71); // Grün für Bindung!
                        atomGruppe.wolkenVisuals[w.id].material.opacity = 0.4;
                    }
                });
            }
        }

        // ========================================================
        // 3. API-VERBINDUNG (DIE BRÜCKE)
        // ========================================================
        async function ladeZustand() {
            try {
                const response = await fetch(`${API_URL}/status`);
                const daten = await response.json();
                document.getElementById('server-status').innerText = "Verbunden";
                document.getElementById('server-status').style.color = "#2ecc71";

                // Szene updaten
                aktualisiere3DSzene(daten);
            } catch (e) {
                document.getElementById('server-status').innerText = "Offline";
                document.getElementById('server-status').style.color = "#e74c3c";
            }
        }

        async function triggereProcrustesBindung() {
            try {
                const response = await fetch(`${API_URL}/binde`, {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({
                        atom_A_id: "O_links", wolken_A_ids: [2, 3],
                        atom_B_id: "O_rechts", wolken_B_ids: [2, 3]
                    })
                });
                await response.json();
                ladeZustand(); // Neu zeichnen nach dem Snap!
            } catch (e) { console.error(e); }
        }

        document.getElementById('btn-binden').addEventListener('click', triggereProcrustesBindung);

        // Neue Funktion, um den Reset-Befehl an Python zu senden
        async function triggereReset() {
            try {
                console.log("Sende Reset-Befehl an das Backend...");
                const response = await fetch(`${API_URL}/reset`, {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' }
                });
                await response.json();

                // WICHTIG: Da die Atome im Backend neu erstellt wurden, müssen wir
                // die alten Three.js-Visualisierungen komplett löschen und neu laden!
                for (const uid in atomMeshes) {
                    scene.remove(atomMeshes[uid]);
                }
                // Speicher im JavaScript leeren
                for (const member in atomMeshes) delete atomMeshes[member];

                // Den frischen Zustand aus Python abrufen und neu zeichnen
                ladeZustand();

            } catch (e) {
                console.error("Reset fehlgeschlagen:", e);
            }
        }

// Event-Listener für den Reset-Button registrieren
document.getElementById('btn-reset').addEventListener('click', triggereReset);
        // Animations-Schleife für flüssiges Rendering
        function animate() {
            requestAnimationFrame(animate);
            controls.update();
            renderer.render(scene, camera);
        }

        // Startschuss
        ladeZustand();
        animate();

        // Fenstergröße dynamisch anpassen
        window.addEventListener('resize', () => {
            camera.aspect = window.innerWidth / window.innerHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(window.innerWidth, window.innerHeight);
        });
    </script>
</body>
</html>